# Governed Text To Reviewed Assertions

Journey purpose: turn raw source text into reviewable candidate assertions, accepted ontology proposals, and overlay-aware report state through the current successor workflow.

Notebook use: `mixed`

This is the canonical journey notebook for the current `onto-canon6` workflow.
It follows the local short-term notebook rules by keeping phase contracts in
`notebooks/notebook_registry.yaml` and rendering each phase explicitly here.

Related docs, code, tests, and evidence:

- `notebooks/notebook_registry.yaml`
- `notebooks/README.md`
- `docs/plans/0001_successor_roadmap.md`
- `docs/STATUS.md`
- `docs/SUCCESSOR_CHARTER.md`
- `docs/adr/0004-keep-text-derived-candidate-assertions-grounded-in-source-evidence-and-route-llm-work-through-llm_client.md`
- `docs/adr/0006-prefer-cli-as-the-first-operational-surface-before-mcp-or-ui.md`
- `docs/adr/0007-adopt-canonical-journey-notebooks-with-an-explicit-registry.md`
- `tests/integration/test_cli_flow.py`
- `tests/integration/test_notebook_process.py`

- `docs/adr/0014-replace-the-v1-semantic-stack-with-pack-driven-canonicalization-and-explicit-recanonicalization.md`


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

SEARCH_ROOTS = [
    Path.cwd().resolve(),
    Path('~/projects/onto-canon6').expanduser(),
]

PROJECT_ROOT = None
for start in SEARCH_ROOTS:
    candidate = start
    while True:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            PROJECT_ROOT = candidate
            break
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    if PROJECT_ROOT is not None:
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate onto-canon6 repo root from notebook cwd or ~/projects/onto-canon6"
    )

for candidate_path in (
    PROJECT_ROOT / "src",
    PROJECT_ROOT.parent / "llm_client",
    Path('~/projects/llm_client').expanduser(),
):
    if candidate_path.exists():
        candidate_str = str(candidate_path)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)

import contextlib
import io
import json
import sqlite3
from pprint import pprint
from tempfile import TemporaryDirectory

from onto_canon6 import cli as cli_module
from onto_canon6.artifacts import ArtifactLineageService
from onto_canon6.extensions.epistemic import EpistemicService
from onto_canon6.notebook_process import load_notebook_registry, validate_notebook_registry
from onto_canon6.ontology_runtime import load_effective_profile, load_profile, validate_assertion_payload
from onto_canon6.pipeline import (
    CandidateAssertionImport,
    EvidenceSpan,
    OverlayApplicationService,
    ProfileRef,
    ReviewService,
    SourceArtifactRef,
)
from onto_canon6.surfaces import EpistemicReportService, LineageReportService


In [2]:
registry_report = validate_notebook_registry()
registry = load_notebook_registry()
journey = registry.journey_by_id('governed_text_to_reviewed_assertions')
phase_by_id = {phase.phase_id: phase for phase in journey.phases}

NOTEBOOK_MODE = 'planning'
artifacts: dict[str, object] = {}
workspace = TemporaryDirectory()
workspace_path = Path(workspace.name)
review_db_path = workspace_path / 'review.sqlite3'
overlay_root = workspace_path / 'ontology_overlays'


def show_phase_contract(phase_id: str) -> None:
    phase = phase_by_id[phase_id]
    pprint(phase.model_dump())


def require_phase_mode(phase_id: str) -> None:
    phase = phase_by_id[phase_id]
    if NOTEBOOK_MODE == 'proof' and phase.proof_critical and phase.execution_mode != 'live':
        raise RuntimeError(
            f"proof mode requires phase '{phase_id}' to run live, got {phase.execution_mode}"
        )


def record_artifact(phase_id: str, artifact: object) -> object:
    artifacts[phase_id] = artifact
    return artifact


def run_cli(args: list[str]) -> tuple[int, str, str]:
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()
    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        exit_code = cli_module.main(args)
    return exit_code, stdout_buffer.getvalue(), stderr_buffer.getvalue()


pprint(registry_report.model_dump())


{'auxiliary_notebook_count': 21,
 'journey_count': 1,
 'phase_count': 15,
 'registry_path': 'notebooks/notebook_registry.yaml',
 'validated_notebooks': ('notebooks/00_master_governed_text_to_reviewed_assertions.ipynb',
                         'notebooks/01_policy_contracts.ipynb',
                         'notebooks/02_donor_profile_loading.ipynb',
                         'notebooks/03_validation_slice.ipynb',
                         'notebooks/04_review_slice.ipynb',
                         'notebooks/05_review_reporting_slice.ipynb',
                         'notebooks/06_overlay_application_slice.ipynb',
                         'notebooks/07_text_grounded_import_contract.ipynb',
                         'notebooks/08_text_extraction_slice.ipynb',
                         'notebooks/09_successor_long_term_plan.ipynb',
                         'notebooks/10_live_extraction_evaluation.ipynb',
                         'notebooks/11_future_phase_breakdown.ipynb',
                   

## Phase 1: Source Artifact Preparation

Purpose: load one real source file and expose it as the explicit source artifact
for the rest of the journey.

Input -> output: `raw_text_file -> source_artifact`

Current gap or promotion path: none. This phase is already live and proven.


In [3]:
phase_id = 'source_artifact_preparation'
require_phase_mode(phase_id)
show_phase_contract(phase_id)

source_path = workspace_path / 'source.txt'
source_text = 'Campaign Alpha used aligned messaging across channels.'
source_path.write_text(source_text, encoding='utf-8')

record_artifact(
    phase_id,
    {
        'artifact_type': 'source_artifact',
        'path': str(source_path),
        'source_kind': 'text_file',
        'source_ref': str(source_path),
        'content_text': source_text,
    },
)
artifacts[phase_id]


{'acceptance_criteria': ('A concrete source file exists and is loaded '
                         'explicitly.',
                         'The downstream workflow receives a named source '
                         'artifact rather than hidden notebook state.',
                         'The phase can run live in the canonical journey '
                         'notebook.'),
 'execution_mode': 'live',
 'input_artifact': 'raw_text_file',
 'output_artifact': 'source_artifact',
 'phase_id': 'source_artifact_preparation',
 'promotion_path': None,
 'proof_critical': True,
 'purpose': 'Load one real source file and expose it as the explicit source '
            'artifact for the rest of the journey.',
 'related_code': ('src/onto_canon6/cli.py',
                  'src/onto_canon6/pipeline/models.py'),
 'related_docs': ('docs/adr/0006-prefer-cli-as-the-first-operational-surface-before-mcp-or-ui.md',),
 'related_evidence': ('notebooks/12_cli_surface.ipynb',),
 'related_tests': ('tests/integration/

{'artifact_type': 'source_artifact',
 'path': '/tmp/tmpsgay7nt1/source.txt',
 'source_kind': 'text_file',
 'source_ref': '/tmp/tmpsgay7nt1/source.txt',
 'content_text': 'Campaign Alpha used aligned messaging across channels.'}

## Phase 2: Candidate Assertion Extraction

Purpose: produce candidate assertions from the source text while keeping source
grounding explicit.

Input -> output: `source_artifact -> candidate_assertion_batch`

Current gap or promotion path: the canonical journey notebook uses an explicit
fixture-shaped extractor stand-in so the notebook remains runnable without a
live model call. Real extraction is separately proved through the linked code,
tests, and evidence notebooks.


In [4]:
phase_id = 'candidate_assertion_extraction'
show_phase_contract(phase_id)


class DeterministicTextExtractionService:
    """Provide a local fixture-shaped stand-in for the remote extraction boundary."""

    def __init__(self, *, review_service: ReviewService) -> None:
        self._review_service = review_service

    def extract_and_submit(
        self,
        *,
        source_text: str,
        profile_id: str,
        profile_version: str,
        submitted_by: str,
        source_ref: str,
        source_kind: str = 'text_file',
        source_label: str | None = None,
        source_metadata: dict[str, object] | None = None,
    ) -> tuple[object, ...]:
        del source_metadata
        phrase = 'aligned messaging'
        start_char = source_text.index(phrase)
        end_char = start_char + len(phrase)
        candidate_import = CandidateAssertionImport(
            profile=ProfileRef(profile_id=profile_id, profile_version=profile_version),
            payload={
                'predicate': 'oc:signals_alignment',
                'roles': {
                    'subject': [
                        {
                            'kind': 'value',
                            'value_kind': 'string',
                            'value': 'Campaign Alpha',
                        }
                    ],
                    'object': [
                        {
                            'kind': 'value',
                            'value_kind': 'string',
                            'value': phrase,
                        }
                    ],
                },
            },
            submitted_by=submitted_by,
            source_artifact=SourceArtifactRef(
                source_kind=source_kind,
                source_ref=source_ref,
                source_label=source_label,
                source_metadata={},
                content_text=source_text,
            ),
            evidence_spans=(
                EvidenceSpan(start_char=start_char, end_char=end_char, text=phrase),
            ),
            claim_text='Campaign Alpha used aligned messaging.',
        )
        return (
            self._review_service.submit_candidate_import(candidate_import=candidate_import),
        )


original_extractor = cli_module.TextExtractionService
cli_module.TextExtractionService = DeterministicTextExtractionService
extract_exit, extract_stdout, extract_stderr = run_cli(
    [
        'extract-text',
        '--input',
        str(source_path),
        '--profile-id',
        'psyop_seed',
        '--profile-version',
        '0.1.0',
        '--submitted-by',
        'analyst:notebook',
        '--review-db-path',
        str(review_db_path),
        '--overlay-root',
        str(overlay_root),
        '--output',
        'json',
    ]
)
assert extract_exit == 0, extract_stderr
extract_result = json.loads(extract_stdout)
record_artifact(phase_id, extract_result)
extract_result


{'acceptance_criteria': ('The phase emits candidate assertions with source '
                         'grounding.',
                         'The canonical journey notebook uses an explicit '
                         'fixture-shaped extractor stand-in rather than hidden '
                         'invention.',
                         'Real extraction code, tests, and benchmark evidence '
                         'remain linked separately.'),
 'execution_mode': 'fixture',
 'input_artifact': 'source_artifact',
 'output_artifact': 'candidate_assertion_batch',
 'phase_id': 'candidate_assertion_extraction',
 'promotion_path': None,
 'proof_critical': False,
 'purpose': 'Produce candidate assertions from raw text while keeping source '
            'grounding explicit.',
 'related_code': ('src/onto_canon6/pipeline/text_extraction.py',
                  'src/onto_canon6/pipeline/service.py',
                  'prompts/extraction/text_to_candidate_assertions.yaml'),
 'related_docs': ('docs/adr

[{'candidate': {'candidate_id': 'cand_6d33c2f0f5ab4f9b82dd0f6b',
   'claim_text': 'Campaign Alpha used aligned messaging.',
   'evidence_spans': [{'end_char': 37,
     'start_char': 20,
     'text': 'aligned messaging'}],
   'normalized_payload': {'predicate': 'oc:signals_alignment',
    'roles': {'object': [{'kind': 'value',
       'value': 'aligned messaging',
       'value_kind': 'string'}],
     'subject': [{'kind': 'value',
       'value': 'Campaign Alpha',
       'value_kind': 'string'}]}},
   'payload': {'predicate': 'oc:signals_alignment',
    'roles': {'object': [{'kind': 'value',
       'value': 'aligned messaging',
       'value_kind': 'string'}],
     'subject': [{'kind': 'value',
       'value': 'Campaign Alpha',
       'value_kind': 'string'}]}},
   'payload_hash': 'sha256:61808a0228906bbaaf1280423471693fa04fa39e993210adeea8b53d4f7e0b71',
   'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
   'proposal_ids': ['prop_22e13cd665e9fe60f13aad1e'],
   'prove

## Phase 3: Candidate Listing And Inspection

Purpose: inspect the persisted candidate assertions through the first
operational surface.

Input -> output: `candidate_assertion_batch -> candidate_listing`

Current gap or promotion path: none. This phase is live and proven.


In [5]:
phase_id = 'candidate_listing_and_inspection'
require_phase_mode(phase_id)
show_phase_contract(phase_id)

list_exit, list_stdout, list_stderr = run_cli(
    [
        'list-candidates',
        '--review-db-path',
        str(review_db_path),
        '--output',
        'json',
    ]
)
assert list_exit == 0, list_stderr
candidate_listing = json.loads(list_stdout)
record_artifact(phase_id, candidate_listing)
candidate_listing


{'acceptance_criteria': ('A user can inspect persisted candidates without '
                         'direct Python calls.',
                         'The listing reflects the persisted review store.',
                         'The phase runs live through the CLI.'),
 'execution_mode': 'live',
 'input_artifact': 'candidate_assertion_batch',
 'output_artifact': 'candidate_listing',
 'phase_id': 'candidate_listing_and_inspection',
 'promotion_path': None,
 'proof_critical': True,
 'purpose': 'Expose pending candidate assertions for explicit inspection '
            'through the first operational surface.',
 'related_code': ('src/onto_canon6/cli.py',
                  'src/onto_canon6/surfaces/review_report.py',
                  'src/onto_canon6/pipeline/service.py'),
 'related_docs': ('docs/STATUS.md',),
 'related_evidence': ('notebooks/05_review_reporting_slice.ipynb',
                      'notebooks/12_cli_surface.ipynb'),
 'related_tests': ('tests/integration/test_cli_flow.py',),
 '

[{'candidate_id': 'cand_6d33c2f0f5ab4f9b82dd0f6b',
  'claim_text': 'Campaign Alpha used aligned messaging.',
  'evidence_spans': [{'end_char': 37,
    'start_char': 20,
    'text': 'aligned messaging'}],
  'normalized_payload': {'predicate': 'oc:signals_alignment',
   'roles': {'object': [{'kind': 'value',
      'value': 'aligned messaging',
      'value_kind': 'string'}],
    'subject': [{'kind': 'value',
      'value': 'Campaign Alpha',
      'value_kind': 'string'}]}},
  'payload': {'predicate': 'oc:signals_alignment',
   'roles': {'object': [{'kind': 'value',
      'value': 'aligned messaging',
      'value_kind': 'string'}],
    'subject': [{'kind': 'value',
      'value': 'Campaign Alpha',
      'value_kind': 'string'}]}},
  'payload_hash': 'sha256:61808a0228906bbaaf1280423471693fa04fa39e993210adeea8b53d4f7e0b71',
  'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
  'proposal_ids': ['prop_22e13cd665e9fe60f13aad1e'],
  'provenance': {'source_artifact': {'conten

## Phase 4: Candidate Review

Purpose: record an explicit accept or reject decision for the candidate.

Input -> output: `candidate_listing -> candidate_review_record`

Current gap or promotion path: none. This phase is live and proven.


In [6]:
phase_id = 'candidate_review'
require_phase_mode(phase_id)
show_phase_contract(phase_id)

candidate_id = extract_result[0]['candidate']['candidate_id']
review_candidate_exit, review_candidate_stdout, review_candidate_stderr = run_cli(
    [
        'review-candidate',
        '--candidate-id',
        candidate_id,
        '--decision',
        'accepted',
        '--actor-id',
        'analyst:reviewer',
        '--review-db-path',
        str(review_db_path),
        '--output',
        'json',
    ]
)
assert review_candidate_exit == 0, review_candidate_stderr
candidate_review_record = json.loads(review_candidate_stdout)
record_artifact(phase_id, candidate_review_record)
candidate_review_record


{'acceptance_criteria': ('Candidate review decisions are explicit and '
                         'persisted.',
                         'Invalid review transitions fail loudly.',
                         'The phase runs live through the CLI.'),
 'execution_mode': 'live',
 'input_artifact': 'candidate_listing',
 'output_artifact': 'candidate_review_record',
 'phase_id': 'candidate_review',
 'promotion_path': None,
 'proof_critical': True,
 'purpose': 'Record an explicit accept or reject decision for the candidate '
            'assertion.',
 'related_code': ('src/onto_canon6/cli.py',
                  'src/onto_canon6/pipeline/service.py',
                  'src/onto_canon6/pipeline/store.py'),
 'related_docs': ('docs/plans/0001_successor_roadmap.md',),
 'related_evidence': ('notebooks/04_review_slice.ipynb',
                      'notebooks/12_cli_surface.ipynb'),
 'related_tests': ('tests/pipeline/test_review_service.py',
                   'tests/integration/test_cli_flow.py'),
 'sta

{'candidate_id': 'cand_6d33c2f0f5ab4f9b82dd0f6b',
 'claim_text': 'Campaign Alpha used aligned messaging.',
 'evidence_spans': [{'end_char': 37,
   'start_char': 20,
   'text': 'aligned messaging'}],
 'normalized_payload': {'predicate': 'oc:signals_alignment',
  'roles': {'object': [{'kind': 'value',
     'value': 'aligned messaging',
     'value_kind': 'string'}],
   'subject': [{'kind': 'value',
     'value': 'Campaign Alpha',
     'value_kind': 'string'}]}},
 'payload': {'predicate': 'oc:signals_alignment',
  'roles': {'object': [{'kind': 'value',
     'value': 'aligned messaging',
     'value_kind': 'string'}],
   'subject': [{'kind': 'value',
     'value': 'Campaign Alpha',
     'value_kind': 'string'}]}},
 'payload_hash': 'sha256:61808a0228906bbaaf1280423471693fa04fa39e993210adeea8b53d4f7e0b71',
 'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
 'proposal_ids': ['prop_22e13cd665e9fe60f13aad1e'],
 'provenance': {'source_artifact': {'content_text': 'Campaign Alph

## Phase 5: Proposal Review

Purpose: accept or reject ontology proposals created by mixed-mode validation.

Input -> output: `candidate_review_record -> proposal_review_record`

Current gap or promotion path: none. This phase is live and proven.


In [7]:
phase_id = 'proposal_review'
require_phase_mode(phase_id)
show_phase_contract(phase_id)

proposal_id = extract_result[0]['proposals'][0]['proposal_id']
review_proposal_exit, review_proposal_stdout, review_proposal_stderr = run_cli(
    [
        'review-proposal',
        '--proposal-id',
        proposal_id,
        '--decision',
        'accepted',
        '--actor-id',
        'analyst:reviewer',
        '--acceptance-policy',
        'apply_to_overlay',
        '--review-db-path',
        str(review_db_path),
        '--output',
        'json',
    ]
)
assert review_proposal_exit == 0, review_proposal_stderr
proposal_review_record = json.loads(review_proposal_stdout)
record_artifact(phase_id, proposal_review_record)
proposal_review_record


{'acceptance_criteria': ('Proposal review remains explicit and separate from '
                         'candidate review.',
                         'Acceptance policy is observable.',
                         'The phase runs live through the CLI.'),
 'execution_mode': 'live',
 'input_artifact': 'candidate_review_record',
 'output_artifact': 'proposal_review_record',
 'phase_id': 'proposal_review',
 'promotion_path': None,
 'proof_critical': True,
 'purpose': 'Accept or reject ontology proposals created by mixed-mode '
            'validation.',
 'related_code': ('src/onto_canon6/cli.py',
                  'src/onto_canon6/pipeline/service.py'),
 'related_docs': ('docs/adr/0006-prefer-cli-as-the-first-operational-surface-before-mcp-or-ui.md',),
 'related_evidence': ('notebooks/06_overlay_application_slice.ipynb',
                      'notebooks/12_cli_surface.ipynb'),
 'related_tests': ('tests/pipeline/test_review_service.py',
                   'tests/integration/test_cli_flow.py'),

{'application_status': 'pending_overlay_apply',
 'candidate_ids': ['cand_6d33c2f0f5ab4f9b82dd0f6b'],
 'created_at': '2026-03-18T21:17:00.382799Z',
 'details': {'candidate_source_kind': 'text_file',
  'source': 'validate_assertion_payload'},
 'overlay_application': None,
 'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
 'proposal_id': 'prop_22e13cd665e9fe60f13aad1e',
 'proposal_key': 'sha256:22e13cd665e9fe60f13aad1e953d210364351edb2239969e344895ea2afaab97',
 'proposal_kind': 'predicate',
 'proposed_value': 'oc:signals_alignment',
 'reason': "default ontology mode 'mixed' resolved action 'propose' for predicate",
 'review': {'acceptance_policy': 'apply_to_overlay',
  'actor_id': 'analyst:reviewer',
  'application_status': 'pending_overlay_apply',
  'created_at': '2026-03-18T21:17:00.451426Z',
  'decision': 'accepted',
  'decision_id': 'dec_640a0eabfd794c2dbb51c39e',
  'note_text': None,
  'proposal_id': 'prop_22e13cd665e9fe60f13aad1e'},
 'status': 'accepted',
 'targe

## Phase 6: Overlay Application

Purpose: apply the accepted ontology proposal into the local overlay without
hiding writeback inside review.

Input -> output: `proposal_review_record -> overlay_application_record`

Current gap or promotion path: none. This phase is live and proven.


In [8]:
phase_id = 'overlay_application'
require_phase_mode(phase_id)
show_phase_contract(phase_id)

apply_exit, apply_stdout, apply_stderr = run_cli(
    [
        'apply-overlay',
        '--proposal-id',
        proposal_id,
        '--actor-id',
        'analyst:reviewer',
        '--review-db-path',
        str(review_db_path),
        '--overlay-root',
        str(overlay_root),
        '--output',
        'json',
    ]
)
assert apply_exit == 0, apply_stderr
overlay_application_record = json.loads(apply_stdout)
record_artifact(phase_id, overlay_application_record)
overlay_application_record


{'acceptance_criteria': ('Accepted proposals can be applied deterministically '
                         'and explicitly.',
                         'Overlay content is inspectable.',
                         'The phase runs live through the CLI.'),
 'execution_mode': 'live',
 'input_artifact': 'proposal_review_record',
 'output_artifact': 'overlay_application_record',
 'phase_id': 'overlay_application',
 'promotion_path': None,
 'proof_critical': True,
 'purpose': 'Apply an accepted ontology proposal into the local overlay '
            'without hiding writeback inside review.',
 'related_code': ('src/onto_canon6/cli.py',
                  'src/onto_canon6/pipeline/overlay_service.py',
                  'src/onto_canon6/ontology_runtime/overlays.py'),
 'related_docs': ('docs/plans/0001_successor_roadmap.md',),
 'related_evidence': ('notebooks/06_overlay_application_slice.ipynb',
                      'notebooks/12_cli_surface.ipynb'),
 'related_tests': ('tests/integration/test_cli_flo

{'application_id': 'oapp_751576168e464a16a5f24651',
 'applied_by': 'analyst:reviewer',
 'applied_value': 'oc:signals_alignment',
 'content_path': '/tmp/tmpsgay7nt1/ontology_overlays/onto_canon_psyop_seed__overlay/0.1.0/predicate_additions.jsonl',
 'created_at': '2026-03-18T21:17:00.476230Z',
 'overlay_pack': {'pack_id': 'onto_canon_psyop_seed__overlay',
  'pack_version': '0.1.0'},
 'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
 'proposal_id': 'prop_22e13cd665e9fe60f13aad1e',
 'proposal_kind': 'predicate'}

## Phase 7: Overlay-Aware Reporting

Purpose: show the resulting accepted proposal and overlay content through a
small report-friendly surface.

Input -> output: `overlay_application_record -> overlay_aware_report`

Current gap or promotion path: none. This phase is live and proven.


In [9]:
phase_id = 'overlay_aware_reporting'
require_phase_mode(phase_id)
show_phase_contract(phase_id)

list_proposals_exit, list_proposals_stdout, list_proposals_stderr = run_cli(
    [
        'list-proposals',
        '--review-db-path',
        str(review_db_path),
        '--status',
        'accepted',
        '--output',
        'json',
    ]
)
assert list_proposals_exit == 0, list_proposals_stderr
overlay_content_path = Path(overlay_application_record['content_path'])
overlay_aware_report = {
    'accepted_proposals': json.loads(list_proposals_stdout),
    'overlay_content': overlay_content_path.read_text(encoding='utf-8'),
}
record_artifact(phase_id, overlay_aware_report)
overlay_aware_report


{'acceptance_criteria': ('Accepted proposals and applied overlay content are '
                         'inspectable together.',
                         'The report stays grounded in persisted state.',
                         'The phase runs live in the canonical journey '
                         'notebook.'),
 'execution_mode': 'live',
 'input_artifact': 'overlay_application_record',
 'output_artifact': 'overlay_aware_report',
 'phase_id': 'overlay_aware_reporting',
 'promotion_path': None,
 'proof_critical': True,
 'purpose': 'Show the resulting accepted proposal and overlay content through '
            'a small report-friendly surface.',
 'related_code': ('src/onto_canon6/cli.py',
                  'src/onto_canon6/surfaces/review_report.py'),
 'related_docs': ('docs/STATUS.md',),
 'related_evidence': ('notebooks/05_review_reporting_slice.ipynb',
                      'notebooks/12_cli_surface.ipynb'),
 'related_tests': ('tests/integration/test_cli_flow.py',),
 'status': 'proven

{'accepted_proposals': [{'application_status': 'applied_to_overlay',
   'candidate_ids': ['cand_6d33c2f0f5ab4f9b82dd0f6b'],
   'created_at': '2026-03-18T21:17:00.382799Z',
   'details': {'candidate_source_kind': 'text_file',
    'source': 'validate_assertion_payload'},
   'overlay_application': {'application_id': 'oapp_751576168e464a16a5f24651',
    'applied_by': 'analyst:reviewer',
    'applied_value': 'oc:signals_alignment',
    'content_path': '/tmp/tmpsgay7nt1/ontology_overlays/onto_canon_psyop_seed__overlay/0.1.0/predicate_additions.jsonl',
    'created_at': '2026-03-18T21:17:00.476230Z',
    'overlay_pack': {'pack_id': 'onto_canon_psyop_seed__overlay',
     'pack_version': '0.1.0'},
    'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
    'proposal_id': 'prop_22e13cd665e9fe60f13aad1e',
    'proposal_kind': 'predicate'},
   'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
   'proposal_id': 'prop_22e13cd665e9fe60f13aad1e',
   'proposal_key': 

## Phase 8: Domain Pack Generalization

Purpose: prove the second-pack workflow using one local DoDAF minimal pack with
strict and mixed profiles over the same vocabulary.

Input -> output: `overlay_aware_report -> second_pack_proof`

Current gap or promotion path: this phase is now live and proven for the local
`dodaf_minimal` slice. The separate deep-dive proof lives in
`notebooks/13_dodaf_minimal_second_pack.ipynb`.


In [10]:
phase_id = 'domain_pack_generalization'
require_phase_mode(phase_id)
show_phase_contract(phase_id)

strict_profile = load_profile('dodaf_minimal_strict', '0.1.0')
mixed_profile = load_profile('dodaf_minimal_mixed', '0.1.0')

strict_known_outcome = validate_assertion_payload(
    {
        'predicate': 'dodaf:operational_node_exchanges_information',
        'roles': {
            'source_node': [
                {
                    'kind': 'entity',
                    'entity_id': 'ent:node:source',
                    'entity_type': 'dm2:OperationalNode',
                }
            ],
            'target_node': [
                {
                    'kind': 'entity',
                    'entity_id': 'ent:node:target',
                    'entity_type': 'dm2:OperationalNode',
                }
            ],
            'information_element': [
                {
                    'kind': 'entity',
                    'entity_id': 'ent:info:message',
                    'entity_type': 'dm2:InformationElement',
                }
            ],
        },
    },
    profile=strict_profile,
)
assert strict_known_outcome.hard_errors == ()

dodaf_review_db_path = workspace_path / 'dodaf_minimal_review.sqlite3'
dodaf_overlay_root = workspace_path / 'dodaf_minimal_overlays'
dodaf_review_service = ReviewService(
    db_path=dodaf_review_db_path,
    overlay_root=dodaf_overlay_root,
)
dodaf_overlay_service = OverlayApplicationService(
    db_path=dodaf_review_db_path,
    overlay_root=dodaf_overlay_root,
)
dodaf_submission = dodaf_review_service.submit_candidate_assertion(
    payload={
        'predicate': 'dodaf:operational_node_supports_activity',
        'roles': {
            'source': [{'kind': 'value', 'value_kind': 'string', 'value': 'Node A'}],
            'target': [{'kind': 'value', 'value_kind': 'string', 'value': 'Activity B'}],
        },
    },
    profile_id='dodaf_minimal_mixed',
    profile_version='0.1.0',
    submitted_by='analyst:notebook',
    source_kind='text_file',
    source_ref='dodaf_source.txt',
)
proposal_id = dodaf_submission.proposals[0].proposal_id
dodaf_review_service.review_proposal(
    proposal_id=proposal_id,
    decision='accepted',
    actor_id='analyst:reviewer',
    acceptance_policy='apply_to_overlay',
)
dodaf_overlay_record = dodaf_overlay_service.apply_proposal_to_overlay(
    proposal_id=proposal_id,
    applied_by='analyst:reviewer',
)
effective_dodaf_profile = load_effective_profile(
    'dodaf_minimal_mixed',
    '0.1.0',
    overlay_root=dodaf_overlay_root,
)

second_pack_proof = {
    'strict_profile_mode': strict_profile.ontology_policy.mode,
    'mixed_profile_mode': mixed_profile.ontology_policy.mode,
    'shared_pack_id': strict_profile.pack_ref.pack_id if strict_profile.pack_ref is not None else None,
    'strict_known_assertion_valid': not strict_known_outcome.has_hard_errors,
    'mixed_submission_validation_status': dodaf_submission.candidate.validation_status,
    'mixed_submission_proposal_count': len(dodaf_submission.proposals),
    'overlay_application_path': dodaf_overlay_record.content_path,
    'active_overlay_predicates': sorted(effective_dodaf_profile.active_overlay_predicates),
}
record_artifact(phase_id, second_pack_proof)
second_pack_proof


{'acceptance_criteria': ('One local second pack loads through the same '
                         'pack/profile machinery as the donor packs.',
                         'A strict profile and a mixed profile share the same '
                         'starting pack vocabulary.',
                         'The same review and overlay workflow works for the '
                         'second pack without core branching.'),
 'execution_mode': 'live',
 'input_artifact': 'overlay_aware_report',
 'output_artifact': 'second_pack_proof',
 'phase_id': 'domain_pack_generalization',
 'promotion_path': None,
 'proof_critical': True,
 'purpose': 'Prove the second-pack workflow using one local DoDAF minimal pack '
            'with strict and mixed profiles over the same vocabulary.',
 'related_code': ('src/onto_canon6/ontology_runtime/loaders.py',
                  'src/onto_canon6/pipeline/service.py',
                  'ontology_packs/dodaf_minimal/0.1.0/manifest.yaml',
                  'profiles/d

{'strict_profile_mode': 'closed',
 'mixed_profile_mode': 'mixed',
 'shared_pack_id': 'dodaf_minimal',
 'strict_known_assertion_valid': True,
 'mixed_submission_validation_status': 'needs_review',
 'mixed_submission_proposal_count': 1,
 'overlay_application_path': '/tmp/tmpsgay7nt1/dodaf_minimal_overlays/dodaf_minimal__overlay/0.1.0/predicate_additions.jsonl',
 'active_overlay_predicates': ['dodaf:operational_node_supports_activity']}

## Phase 9: Artifact Lineage Recovery

Purpose: recover artifact-backed provenance through a narrow three-kind model
without rebuilding a fused runtime.

Input -> output: `second_pack_proof -> artifact_lineage_proof`

Current gap or promotion path: none. This phase is now live and proven for the
narrow candidate-centered lineage slice. The separate deep-dive proof lives in
`notebooks/14_artifact_lineage_slice.ipynb`.


In [11]:
phase_id = 'artifact_lineage_recovery'
require_phase_mode(phase_id)
show_phase_contract(phase_id)

artifact_service = ArtifactLineageService(db_path=review_db_path)
lineage_report_service = LineageReportService(artifact_service=artifact_service)

source_artifact = artifact_service.register_artifact(
    artifact_kind='source',
    uri=str(source_path),
    label='canonical journey source text',
    metadata={'source_kind': 'text_file'},
)
derived_dataset = artifact_service.register_artifact(
    artifact_kind='derived_dataset',
    uri='derived/alignment_dataset.jsonl',
    label='alignment dataset',
    metadata={'transform': 'candidate aggregation'},
)
analysis_result = artifact_service.register_artifact(
    artifact_kind='analysis_result',
    uri='analysis/alignment_scores.json',
    label='alignment score report',
    metadata={'metric': 'alignment_score', 'score': 0.93},
)

artifact_service.add_lineage_edge(
    parent_artifact_id=source_artifact.artifact_id,
    child_artifact_id=derived_dataset.artifact_id,
)
artifact_service.add_lineage_edge(
    parent_artifact_id=derived_dataset.artifact_id,
    child_artifact_id=analysis_result.artifact_id,
)
artifact_service.link_candidate_artifact(
    candidate_id=candidate_id,
    artifact_id=source_artifact.artifact_id,
    support_kind='quoted_from',
    reference_detail='aligned messaging source text',
)
artifact_service.link_candidate_artifact(
    candidate_id=candidate_id,
    artifact_id=analysis_result.artifact_id,
    support_kind='supported_by_analysis',
    reference_detail='alignment score 0.93',
)

artifact_lineage_proof = lineage_report_service.build_candidate_report(
    candidate_id=candidate_id,
).model_dump()
record_artifact(phase_id, artifact_lineage_proof)
artifact_lineage_proof


{'acceptance_criteria': ('Typed artifact records with explicit kinds are '
                         'persisted.',
                         'Candidate-centered artifact links and lineage edges '
                         'are inspectable through a live report.',
                         'The phase runs live in the canonical journey '
                         'notebook.'),
 'execution_mode': 'live',
 'input_artifact': 'second_pack_proof',
 'output_artifact': 'artifact_lineage_proof',
 'phase_id': 'artifact_lineage_recovery',
 'promotion_path': None,
 'proof_critical': True,
 'purpose': 'Recover artifact-backed provenance through a narrow three-kind '
            'model without rebuilding a fused runtime.',
 'related_code': ('src/onto_canon6/artifacts/models.py',
                  'src/onto_canon6/artifacts/store.py',
                  'src/onto_canon6/artifacts/service.py',
                  'src/onto_canon6/surfaces/lineage_report.py'),
 'related_docs': ('docs/plans/0001_successor_roadma

{'candidate': {'candidate_id': 'cand_6d33c2f0f5ab4f9b82dd0f6b',
  'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
  'validation_status': 'needs_review',
  'review_status': 'accepted',
  'payload_hash': 'sha256:61808a0228906bbaaf1280423471693fa04fa39e993210adeea8b53d4f7e0b71',
  'payload': {'predicate': 'oc:signals_alignment',
   'roles': {'object': [{'kind': 'value',
      'value': 'aligned messaging',
      'value_kind': 'string'}],
    'subject': [{'kind': 'value',
      'value': 'Campaign Alpha',
      'value_kind': 'string'}]}},
  'normalized_payload': {'predicate': 'oc:signals_alignment',
   'roles': {'object': [{'kind': 'value',
      'value': 'aligned messaging',
      'value_kind': 'string'}],
    'subject': [{'kind': 'value',
      'value': 'Campaign Alpha',
      'value_kind': 'string'}]}},
  'validation': {'hard_errors': (),
   'soft_violations': ({'code': 'oc:profile_unknown_predicate',
     'message': "predicate 'oc:signals_alignment' is not declared i

## Phase 10: Epistemic Extension

Purpose: recover the first optional epistemic slice without moving epistemic
policy into the core workflow.

Input -> output: `artifact_lineage_proof -> epistemic_extension_proof`

Current gap or promotion path: none. This phase is now live and proven for the
first extension-local confidence and supersession slice. The separate deep-dive
proof lives in `notebooks/15_epistemic_extension_slice.ipynb`.


In [12]:
phase_id = 'epistemic_extension'
require_phase_mode(phase_id)
show_phase_contract(phase_id)

review_service = ReviewService(db_path=review_db_path, overlay_root=overlay_root)

replacement_submission = review_service.submit_candidate_assertion(
    payload={
        'predicate': 'oc:signals_alignment_replacement',
        'roles': {
            'subject': [
                {
                    'kind': 'value',
                    'value_kind': 'string',
                    'value': 'Campaign Alpha',
                }
            ],
            'object': [
                {
                    'kind': 'value',
                    'value_kind': 'string',
                    'value': 'coordinated messaging',
                }
            ],
        },
    },
    profile_id='default',
    profile_version='1.0.0',
    submitted_by='analyst:epistemic-notebook',
    source_kind='text_note',
    source_ref='notes/replacement_alignment.txt',
    source_label='replacement alignment note',
    source_text='Campaign Alpha used coordinated messaging across channels.',
)
replacement_candidate = review_service.review_candidate(
    candidate_id=replacement_submission.candidate.candidate_id,
    decision='accepted',
    actor_id='analyst:epistemic-reviewer',
)

epistemic_service = EpistemicService(db_path=review_db_path)
epistemic_service.record_confidence(
    candidate_id=candidate_id,
    confidence_score=0.78,
    source_kind='user',
    actor_id='analyst:epistemic-reviewer',
    rationale='Initial reviewed claim is well supported but broad.',
)
epistemic_service.record_confidence(
    candidate_id=replacement_candidate.candidate_id,
    confidence_score=0.91,
    source_kind='user',
    actor_id='analyst:epistemic-reviewer',
    rationale='Replacement claim is narrower and better supported.',
)
epistemic_service.record_supersession(
    prior_candidate_id=candidate_id,
    replacement_candidate_id=replacement_candidate.candidate_id,
    actor_id='analyst:epistemic-reviewer',
    rationale='Replacement claim supersedes the broader original wording.',
)

epistemic_extension_proof = EpistemicReportService(
    epistemic_service=epistemic_service,
).build_candidate_report(candidate_id=candidate_id).model_dump()
record_artifact(phase_id, epistemic_extension_proof)
epistemic_extension_proof


{'acceptance_criteria': ('Confidence and supersession are persisted through an '
                         'extension-local package.',
                         'Epistemic state attaches only to accepted candidates '
                         'and remains outside the base review schema.',
                         'The phase runs live in the canonical journey '
                         'notebook.'),
 'execution_mode': 'live',
 'input_artifact': 'artifact_lineage_proof',
 'output_artifact': 'epistemic_extension_proof',
 'phase_id': 'epistemic_extension',
 'promotion_path': None,
 'proof_critical': True,
 'purpose': 'Recover the first optional epistemic slice without moving '
            'epistemic policy into the core workflow.',
 'related_code': ('src/onto_canon6/extensions/epistemic/models.py',
                  'src/onto_canon6/extensions/epistemic/store.py',
                  'src/onto_canon6/extensions/epistemic/service.py',
                  'src/onto_canon6/surfaces/epistemic_report.

{'candidate': {'candidate_id': 'cand_6d33c2f0f5ab4f9b82dd0f6b',
  'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
  'validation_status': 'needs_review',
  'review_status': 'accepted',
  'payload_hash': 'sha256:61808a0228906bbaaf1280423471693fa04fa39e993210adeea8b53d4f7e0b71',
  'payload': {'predicate': 'oc:signals_alignment',
   'roles': {'object': [{'kind': 'value',
      'value': 'aligned messaging',
      'value_kind': 'string'}],
    'subject': [{'kind': 'value',
      'value': 'Campaign Alpha',
      'value_kind': 'string'}]}},
  'normalized_payload': {'predicate': 'oc:signals_alignment',
   'roles': {'object': [{'kind': 'value',
      'value': 'aligned messaging',
      'value_kind': 'string'}],
    'subject': [{'kind': 'value',
      'value': 'Campaign Alpha',
      'value_kind': 'string'}]}},
  'validation': {'hard_errors': (),
   'soft_violations': ({'code': 'oc:profile_unknown_predicate',
     'message': "predicate 'oc:signals_alignment' is not declared i

## Phase 11: Product-Facing Workflow Integration

Purpose: export one governed bundle through the CLI-backed product workflow so the successor ends in a real downstream artifact, not just notebook inspection.

Input -> output: `epistemic_extension_proof -> governed_bundle_proof`

Current gap or promotion path: none. This phase is now live and proved through the dedicated governed-bundle workflow notebook in `notebooks/16_governed_bundle_workflow.ipynb`.

In [13]:
phase_id = 'product_facing_workflow_integration'
require_phase_mode(phase_id)
show_phase_contract(phase_id)

export_exit, export_stdout, export_stderr = run_cli(
    [
        'export-governed-bundle',
        '--review-db-path',
        str(review_db_path),
        '--output',
        'json',
    ]
)
assert export_exit == 0, export_stderr
governed_bundle_proof = json.loads(export_stdout)
record_artifact(phase_id, governed_bundle_proof)
governed_bundle_proof


{'acceptance_criteria': ('The workflow runs end to end without direct '
                         'module-level Python calls.',
                         'The exported bundle includes accepted candidates, '
                         'governance state, and provenance.',
                         'The phase runs live in the canonical journey '
                         'notebook.'),
 'execution_mode': 'live',
 'input_artifact': 'epistemic_extension_proof',
 'output_artifact': 'governed_bundle_proof',
 'phase_id': 'product_facing_workflow_integration',
 'promotion_path': None,
 'proof_critical': True,
 'purpose': 'Export one governed bundle through the CLI-backed product '
            'workflow.',
 'related_code': ('src/onto_canon6/cli.py',
                  'src/onto_canon6/surfaces/governed_bundle.py'),
 'related_docs': ('docs/plans/0001_successor_roadmap.md',
                  'docs/plans/0004_phase10_governed_bundle_shape.md',
                  'docs/adr/0010-choose-cli-driven-governed-bun

{'candidate_bundles': [{'artifact_links': [{'artifact_id': 'art_220bf150e1334cb8ba4ec8f1',
     'candidate_id': 'cand_6d33c2f0f5ab4f9b82dd0f6b',
     'created_at': '2026-03-18T21:17:00.830180+00:00',
     'reference_detail': 'aligned messaging source text',
     'support_kind': 'quoted_from'},
    {'artifact_id': 'art_39baa5c9b5e44009ae02e24c',
     'candidate_id': 'cand_6d33c2f0f5ab4f9b82dd0f6b',
     'created_at': '2026-03-18T21:17:00.838779+00:00',
     'reference_detail': 'alignment score 0.93',
     'support_kind': 'supported_by_analysis'}],
   'artifacts': [{'artifact_id': 'art_220bf150e1334cb8ba4ec8f1',
     'artifact_kind': 'source',
     'created_at': '2026-03-18T21:17:00.792851+00:00',
     'fingerprint': None,
     'label': 'canonical journey source text',
     'metadata': {'source_kind': 'text_file'},
     'uri': '/tmp/tmpsgay7nt1/source.txt'},
    {'artifact_id': 'art_04410a9f535343bea1900c78',
     'artifact_kind': 'derived_dataset',
     'created_at': '2026-03-18T21:17:0

## Phase 12: Canonical Graph Promotion

Purpose: promote one accepted candidate into the durable graph layer and inspect the candidate-backed promoted graph report.

Input -> output: `governed_bundle_proof -> promoted_graph_proof`

Current gap or promotion path: none. This phase is now live and proved through the dedicated canonical-graph recovery notebook in `notebooks/17_canonical_graph_recovery_slice.ipynb`.

In [14]:
phase_id = 'canonical_graph_promotion'
require_phase_mode(phase_id)
show_phase_contract(phase_id)

promote_exit, promote_stdout, promote_stderr = run_cli(
    [
        'promote-candidate',
        '--candidate-id',
        candidate_id,
        '--actor-id',
        'analyst:graph-promoter',
        '--review-db-path',
        str(review_db_path),
        '--output',
        'json',
    ]
)
assert promote_exit == 0, promote_stderr
promotion_result = json.loads(promote_stdout)

list_promoted_exit, list_promoted_stdout, list_promoted_stderr = run_cli(
    [
        'list-promoted-assertions',
        '--review-db-path',
        str(review_db_path),
        '--output',
        'json',
    ]
)
assert list_promoted_exit == 0, list_promoted_stderr
promoted_assertions = json.loads(list_promoted_stdout)

graph_report_exit, graph_report_stdout, graph_report_stderr = run_cli(
    [
        'export-promoted-graph-report',
        '--review-db-path',
        str(review_db_path),
        '--output',
        'json',
    ]
)
assert graph_report_exit == 0, graph_report_stderr
promoted_graph_proof = {
    'promotion_result': json.loads(promote_stdout),
    'promoted_assertions': promoted_assertions,
    'report': json.loads(graph_report_stdout),
}
record_artifact(phase_id, promoted_graph_proof)
promoted_graph_proof

{'acceptance_criteria': ('Accepted candidates can be promoted into '
                         'deterministic durable graph records.',
                         'Promoted assertions remain explicitly linked back to '
                         'the source candidate and its governance context.',
                         'The promoted graph state is inspectable through a '
                         'thin CLI-backed report surface.'),
 'execution_mode': 'live',
 'input_artifact': 'governed_bundle_proof',
 'output_artifact': 'promoted_graph_proof',
 'phase_id': 'canonical_graph_promotion',
 'promotion_path': None,
 'proof_critical': True,
 'purpose': 'Promote one accepted candidate into the durable graph layer and '
            'inspect the candidate-backed promoted graph report.',
 'related_code': ('src/onto_canon6/core/graph_models.py',
                  'src/onto_canon6/core/graph_store.py',
                  'src/onto_canon6/core/graph_service.py',
                  'src/onto_canon6/surface

{'promotion_result': {'assertion': {'assertion_id': 'gassert_184dd39bebd9da0a1c1f725b',
   'claim_text': 'Campaign Alpha used aligned messaging.',
   'normalized_body': {'predicate': 'oc:signals_alignment',
    'roles': {'object': [{'kind': 'value',
       'value': 'aligned messaging',
       'value_kind': 'string'}],
     'subject': [{'kind': 'value',
       'value': 'Campaign Alpha',
       'value_kind': 'string'}]}},
   'predicate': 'oc:signals_alignment',
   'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
   'promoted_at': '2026-03-18T21:17:01.070191+00:00',
   'promoted_by': 'analyst:graph-promoter',
   'source_candidate_id': 'cand_6d33c2f0f5ab4f9b82dd0f6b'},
  'entities': [],
  'role_fillers': [{'assertion_id': 'gassert_184dd39bebd9da0a1c1f725b',
    'entity_id': None,
    'entity_type': None,
    'filler_index': 0,
    'filler_kind': 'value',
    'role_id': 'object',
    'value': 'aligned messaging',
    'value_kind': 'string'},
   {'assertion_id': 'gassert_

## Phase 13: Stable Identity And External References

Purpose: create one stable identity over promoted entities, attach alias membership, and record explicit external-reference state.

Input -> output: `promoted_graph_proof -> stable_identity_proof`

Current gap or promotion path: none. This phase is now live and proved through the dedicated identity notebook in `notebooks/18_stable_identity_slice.ipynb`.

In [15]:
phase_id = 'stable_identity_and_external_references'
require_phase_mode(phase_id)
show_phase_contract(phase_id)

identity_submission_one = review_service.submit_candidate_assertion(
    payload={
        'predicate': 'oc:identity_journey_one',
        'roles': {
            'subject': [
                {'entity_id': 'ent:person:eric_olson', 'entity_type': 'oc:person'}
            ],
            'descriptor': [
                {'kind': 'value', 'value_kind': 'string', 'value': 'identity journey demo'}
            ],
        },
    },
    profile_id='default',
    profile_version='1.0.0',
    submitted_by='analyst:identity-journey',
    source_kind='text_note',
    source_ref='notes/identity_journey_one.txt',
    source_text='Identity journey source one.',
)
identity_candidate_one = review_service.review_candidate(
    candidate_id=identity_submission_one.candidate.candidate_id,
    decision='accepted',
    actor_id='analyst:identity-journey',
)
identity_submission_two = review_service.submit_candidate_assertion(
    payload={
        'predicate': 'oc:identity_journey_two',
        'roles': {
            'subject': [
                {'entity_id': 'ent:person:admiral_eric_olson', 'entity_type': 'oc:person'}
            ],
            'descriptor': [
                {'kind': 'value', 'value_kind': 'string', 'value': 'identity journey demo'}
            ],
        },
    },
    profile_id='default',
    profile_version='1.0.0',
    submitted_by='analyst:identity-journey',
    source_kind='text_note',
    source_ref='notes/identity_journey_two.txt',
    source_text='Identity journey source two.',
)
identity_candidate_two = review_service.review_candidate(
    candidate_id=identity_submission_two.candidate.candidate_id,
    decision='accepted',
    actor_id='analyst:identity-journey',
)
promote_identity_one_exit, _, promote_identity_one_stderr = run_cli([
    'promote-candidate', '--candidate-id', identity_candidate_one.candidate_id, '--actor-id', 'analyst:graph-promoter', '--review-db-path', str(review_db_path), '--output', 'json'
])
assert promote_identity_one_exit == 0, promote_identity_one_stderr
promote_identity_two_exit, _, promote_identity_two_stderr = run_cli([
    'promote-candidate', '--candidate-id', identity_candidate_two.candidate_id, '--actor-id', 'analyst:graph-promoter', '--review-db-path', str(review_db_path), '--output', 'json'
])
assert promote_identity_two_exit == 0, promote_identity_two_stderr

create_identity_exit, create_identity_stdout, create_identity_stderr = run_cli([
    'create-identity-for-entity', '--entity-id', 'ent:person:eric_olson', '--actor-id', 'analyst:identity', '--display-label', 'Eric Olson', '--review-db-path', str(review_db_path), '--output', 'json'
])
assert create_identity_exit == 0, create_identity_stderr
identity_bundle = json.loads(create_identity_stdout)
identity_id = identity_bundle['identity']['identity_id']
alias_exit, _, alias_stderr = run_cli([
    'attach-identity-alias', '--identity-id', identity_id, '--entity-id', 'ent:person:admiral_eric_olson', '--actor-id', 'analyst:identity', '--review-db-path', str(review_db_path), '--output', 'json'
])
assert alias_exit == 0, alias_stderr
attach_ref_exit, _, attach_ref_stderr = run_cli([
    'attach-external-ref', '--identity-id', identity_id, '--provider', 'wikidata', '--external-id', 'Q5388397', '--reference-label', 'Eric Olson', '--actor-id', 'analyst:identity', '--review-db-path', str(review_db_path), '--output', 'json'
])
assert attach_ref_exit == 0, attach_ref_stderr
unresolved_ref_exit, _, unresolved_ref_stderr = run_cli([
    'record-unresolved-external-ref', '--identity-id', identity_id, '--provider', 'wikidata', '--unresolved-note', 'Possible second profile needs review', '--actor-id', 'analyst:identity', '--review-db-path', str(review_db_path), '--output', 'json'
])
assert unresolved_ref_exit == 0, unresolved_ref_stderr
identity_report_exit, identity_report_stdout, identity_report_stderr = run_cli([
    'export-identity-report', '--review-db-path', str(review_db_path), '--output', 'json'
])
assert identity_report_exit == 0, identity_report_stderr
stable_identity_proof = json.loads(identity_report_stdout)
record_artifact(phase_id, stable_identity_proof)
stable_identity_proof

{'acceptance_criteria': ('Repeated identity creation for the same promoted '
                         'entity reuses the same local identity.',
                         'Alias membership is explicit and auditable.',
                         'External references are persisted as explicit '
                         'attached or unresolved records.'),
 'execution_mode': 'live',
 'input_artifact': 'promoted_graph_proof',
 'output_artifact': 'stable_identity_proof',
 'phase_id': 'stable_identity_and_external_references',
 'promotion_path': None,
 'proof_critical': True,
 'purpose': 'Create stable identities over promoted entities, attach alias '
            'membership, and record explicit external-reference state.',
 'related_code': ('src/onto_canon6/core/identity_models.py',
                  'src/onto_canon6/core/identity_store.py',
                  'src/onto_canon6/core/identity_service.py',
                  'src/onto_canon6/surfaces/identity_report.py',
                  'src/onto_ca

{'identity_bundles': [{'external_references': [{'attached_at': '2026-03-18T21:17:01.280539+00:00',
     'attached_by': 'analyst:identity',
     'external_id': 'Q5388397',
     'identity_id': 'gid_d9199a02acfe3e5ab6c8a0b1',
     'provider': 'wikidata',
     'reference_id': 'gref_978a743ad08848f497aaf2dd',
     'reference_label': 'Eric Olson',
     'reference_status': 'attached',
     'unresolved_note': None},
    {'attached_at': '2026-03-18T21:17:01.297547+00:00',
     'attached_by': 'analyst:identity',
     'external_id': None,
     'identity_id': 'gid_d9199a02acfe3e5ab6c8a0b1',
     'provider': 'wikidata',
     'reference_id': 'gref_aef5157058934778a1a674b7',
     'reference_label': None,
     'reference_status': 'unresolved',
     'unresolved_note': 'Possible second profile needs review'}],
   'identity': {'created_at': '2026-03-18T21:17:01.255964+00:00',
    'created_by': 'analyst:identity',
    'display_label': 'Eric Olson',
    'identity_id': 'gid_d9199a02acfe3e5ab6c8a0b1',
    'i

## Phase 14: Semantic Canonicalization Repair

Purpose: repair one promoted assertion through the explicit pack-driven semantic canonicalization seam.

Input -> output: `stable_identity_proof -> semantic_canonicalization_proof`

Current gap or promotion path: none. This phase is now live and proved through the dedicated semantic-canonicalization notebook in `notebooks/19_semantic_canonicalization_slice.ipynb`.


In [16]:
phase_id = 'semantic_canonicalization_repair'
require_phase_mode(phase_id)
show_phase_contract(phase_id)

semantic_submission = review_service.submit_candidate_assertion(
    payload={
        'predicate': 'dodaf:operational_node_exchanges_information',
        'roles': {
            'source_node': [
                {'entity_id': 'ent:node:journey_source', 'entity_type': 'dm2:OperationalNode'}
            ],
            'target_node': [
                {'entity_id': 'ent:node:journey_target', 'entity_type': 'dm2:OperationalNode'}
            ],
            'information_element': [
                {'entity_id': 'ent:info:journey_message', 'entity_type': 'dm2:InformationElement'}
            ],
        },
    },
    profile_id='dodaf_minimal_strict',
    profile_version='0.1.0',
    submitted_by='analyst:semantic-journey',
    source_kind='text_note',
    source_ref='notes/semantic_journey.txt',
    source_text='Node A exchanges Message M with Node B.',
)
semantic_candidate = review_service.review_candidate(
    candidate_id=semantic_submission.candidate.candidate_id,
    decision='accepted',
    actor_id='analyst:semantic-journey',
)
promote_semantic_exit, promote_semantic_stdout, promote_semantic_stderr = run_cli([
    'promote-candidate', '--candidate-id', semantic_candidate.candidate_id, '--actor-id', 'analyst:graph-promoter', '--review-db-path', str(review_db_path), '--output', 'json'
])
assert promote_semantic_exit == 0, promote_semantic_stderr
semantic_promotion = json.loads(promote_semantic_stdout)
semantic_assertion_id = semantic_promotion['assertion']['assertion_id']

with sqlite3.connect(review_db_path) as conn:
    conn.execute(
        '''
        UPDATE promoted_graph_assertions
        SET predicate = ?,
            normalized_body_json = ?
        WHERE assertion_id = ?
        ''',
        (
            'OperationalNodeExchangesInformation',
            json.dumps(
                {
                    'predicate': 'OperationalNodeExchangesInformation',
                    'roles': {
                        'source': [
                            {'kind': 'entity', 'entity_id': 'ent:node:journey_source', 'entity_type': 'dm2:OperationalNode'}
                        ],
                        'target': [
                            {'kind': 'entity', 'entity_id': 'ent:node:journey_target', 'entity_type': 'dm2:OperationalNode'}
                        ],
                        'information': [
                            {'kind': 'entity', 'entity_id': 'ent:info:journey_message', 'entity_type': 'dm2:InformationElement'}
                        ],
                    },
                },
                sort_keys=True,
            ),
            semantic_assertion_id,
        ),
    )
    conn.execute(
        '''
        UPDATE promoted_graph_role_fillers
        SET role_id = CASE role_id
            WHEN 'source_node' THEN 'source'
            WHEN 'target_node' THEN 'target'
            WHEN 'information_element' THEN 'information'
            ELSE role_id
        END
        WHERE assertion_id = ?
        ''',
        (semantic_assertion_id,),
    )

repair_exit, repair_stdout, repair_stderr = run_cli([
    'recanonicalize-promoted-assertion', '--assertion-id', semantic_assertion_id, '--actor-id', 'analyst:semantic-repair', '--reason', 'Normalize journey alias ids.', '--review-db-path', str(review_db_path), '--output', 'json'
])
assert repair_exit == 0, repair_stderr
repair_result = json.loads(repair_stdout)
report_exit, report_stdout, report_stderr = run_cli([
    'export-semantic-canonicalization-report', '--review-db-path', str(review_db_path), '--output', 'json'
])
assert report_exit == 0, report_stderr
semantic_canonicalization_proof = {
    'repair_result': repair_result,
    'report': json.loads(report_stdout),
    'before_exact_canonical_match': 0,
    'after_exact_canonical_match': int(repair_result['assertion']['predicate'] == 'dodaf:operational_node_exchanges_information'),
}
record_artifact(phase_id, semantic_canonicalization_proof)
semantic_canonicalization_proof


{'acceptance_criteria': ('Bad promoted graph state can be repaired '
                         'deterministically through pack-declared '
                         'canonicalization metadata.',
                         'Repaired promoted assertions are revalidated before '
                         'persistence.',
                         'The repair trail is persisted and inspectable '
                         'through a thin CLI-backed report surface.'),
 'execution_mode': 'live',
 'input_artifact': 'stable_identity_proof',
 'output_artifact': 'semantic_canonicalization_proof',
 'phase_id': 'semantic_canonicalization_repair',
 'promotion_path': None,
 'proof_critical': True,
 'purpose': 'Repair one promoted assertion through pack-driven predicate and '
            'role canonicalization with explicit persisted recanonicalization '
            'events.',
 'related_code': ('src/onto_canon6/ontology_runtime/loaders.py',
                  'src/onto_canon6/core/semantic_models.py',
         

{'repair_result': {'assertion': {'assertion_id': 'gassert_116dde031f033426120b95e5',
   'claim_text': None,
   'normalized_body': {'predicate': 'dodaf:operational_node_exchanges_information',
    'roles': {'information_element': [{'entity_id': 'ent:info:journey_message',
       'entity_type': 'dm2:InformationElement',
       'kind': 'entity'}],
     'source_node': [{'entity_id': 'ent:node:journey_source',
       'entity_type': 'dm2:OperationalNode',
       'kind': 'entity'}],
     'target_node': [{'entity_id': 'ent:node:journey_target',
       'entity_type': 'dm2:OperationalNode',
       'kind': 'entity'}]}},
   'predicate': 'dodaf:operational_node_exchanges_information',
   'profile': {'profile_id': 'dodaf_minimal_strict',
    'profile_version': '0.1.0'},
   'promoted_at': '2026-03-18T21:17:01.348868+00:00',
   'promoted_by': 'analyst:graph-promoter',
   'source_candidate_id': 'cand_7595d5d5eaf04569bdce1b12'},
  'event': {'actor_id': 'analyst:semantic-repair',
   'after_body': {'predi

## Phase 15: Promoted Assertion Epistemics

In [17]:
phase_id = 'promoted_assertion_epistemics'
require_phase_mode(phase_id)
show_phase_contract(phase_id)

corroboration_submission_a = review_service.submit_candidate_assertion(
    payload={
        'predicate': 'oc:hold_command_role',
        'roles': {
            'commander': [{'entity_id': 'ent:person:journey_cor_a'}],
            'organization': [{'entity_id': 'ent:org:journey_cor_org'}],
            'title': [{'kind': 'value', 'value_kind': 'string', 'value': 'Commander'}],
        },
    },
    profile_id='default',
    profile_version='1.0.0',
    submitted_by='analyst:epistemic-journey',
    source_kind='text_note',
    source_ref='notes/epistemic_journey_cor_a.txt',
    source_text='Journey corroboration source A.',
)
corroboration_candidate_a = review_service.review_candidate(
    candidate_id=corroboration_submission_a.candidate.candidate_id,
    decision='accepted',
    actor_id='analyst:epistemic-journey',
)
corroboration_submission_b = review_service.submit_candidate_assertion(
    payload={
        'predicate': 'oc:hold_command_role',
        'roles': {
            'commander': [{'entity_id': 'ent:person:journey_cor_a'}],
            'organization': [{'entity_id': 'ent:org:journey_cor_org'}],
            'title': [{'kind': 'value', 'value_kind': 'string', 'value': 'Commander'}],
        },
    },
    profile_id='default',
    profile_version='1.0.0',
    submitted_by='analyst:epistemic-journey',
    source_kind='text_note',
    source_ref='notes/epistemic_journey_cor_b.txt',
    source_text='Journey corroboration source B.',
)
corroboration_candidate_b = review_service.review_candidate(
    candidate_id=corroboration_submission_b.candidate.candidate_id,
    decision='accepted',
    actor_id='analyst:epistemic-journey',
)
tension_submission_a = review_service.submit_candidate_assertion(
    payload={
        'predicate': 'oc:hold_command_role',
        'roles': {
            'commander': [{'entity_id': 'ent:person:journey_tension'}],
            'organization': [{'entity_id': 'ent:org:journey_tension_org'}],
            'title': [{'kind': 'value', 'value_kind': 'string', 'value': 'Commander'}],
        },
    },
    profile_id='default',
    profile_version='1.0.0',
    submitted_by='analyst:epistemic-journey',
    source_kind='text_note',
    source_ref='notes/epistemic_journey_tension_a.txt',
    source_text='Journey tension source A.',
)
tension_candidate_a = review_service.review_candidate(
    candidate_id=tension_submission_a.candidate.candidate_id,
    decision='accepted',
    actor_id='analyst:epistemic-journey',
)
tension_submission_b = review_service.submit_candidate_assertion(
    payload={
        'predicate': 'oc:hold_command_role',
        'roles': {
            'commander': [{'entity_id': 'ent:person:journey_tension'}],
            'organization': [{'entity_id': 'ent:org:journey_tension_org'}],
            'title': [{'kind': 'value', 'value_kind': 'string', 'value': 'Director'}],
        },
    },
    profile_id='default',
    profile_version='1.0.0',
    submitted_by='analyst:epistemic-journey',
    source_kind='text_note',
    source_ref='notes/epistemic_journey_tension_b.txt',
    source_text='Journey tension source B.',
)
tension_candidate_b = review_service.review_candidate(
    candidate_id=tension_submission_b.candidate.candidate_id,
    decision='accepted',
    actor_id='analyst:epistemic-journey',
)

phase15_promotions = []
for promoted_candidate_id in (
    corroboration_candidate_a.candidate_id,
    corroboration_candidate_b.candidate_id,
    tension_candidate_a.candidate_id,
    tension_candidate_b.candidate_id,
):
    promote_exit, promote_stdout, promote_stderr = run_cli([
        'promote-candidate', '--candidate-id', promoted_candidate_id, '--actor-id', 'analyst:graph-promoter', '--review-db-path', str(review_db_path), '--output', 'json'
    ])
    assert promote_exit == 0, promote_stderr
    phase15_promotions.append(json.loads(promote_stdout))

disposition_exit, disposition_stdout, disposition_stderr = run_cli([
    'record-assertion-disposition',
    '--assertion-id',
    phase15_promotions[2]['assertion']['assertion_id'],
    '--target-status',
    'weakened',
    '--actor-id',
    'analyst:epistemic-journey',
    '--reason',
    'Conflicting title evidence lowers certainty but does not retract the claim.',
    '--review-db-path',
    str(review_db_path),
    '--output',
    'json',
])
assert disposition_exit == 0, disposition_stderr

report_exit, report_stdout, report_stderr = run_cli([
    'export-assertion-epistemic-report', '--review-db-path', str(review_db_path), '--output', 'json'
])
assert report_exit == 0, report_stderr
promoted_assertion_epistemic_proof = {
    'promotions': phase15_promotions,
    'disposition': json.loads(disposition_stdout),
    'report': json.loads(report_stdout),
    'temporal_inference_disposition': 'deferred',
}
record_artifact(phase_id, promoted_assertion_epistemic_proof)
promoted_assertion_epistemic_proof


{'acceptance_criteria': ('Promoted assertions can move through explicit '
                         'extension-local disposition history beyond '
                         'supersession alone.',
                         'Corroboration and tension are inspectable through a '
                         'typed report surface over promoted graph state.',
                         'The phase runs live in the canonical journey '
                         'notebook.'),
 'execution_mode': 'live',
 'input_artifact': 'semantic_canonicalization_proof',
 'output_artifact': 'promoted_assertion_epistemic_proof',
 'phase_id': 'promoted_assertion_epistemics',
 'promotion_path': None,
 'proof_critical': True,
 'purpose': 'Apply promoted-assertion dispositions and inspect derived '
            'corroboration and tension reports without relocating epistemic '
            'policy into core.',
 'related_code': ('src/onto_canon6/extensions/epistemic/models.py',
                  'src/onto_canon6/extensions/episte

{'promotions': [{'assertion': {'assertion_id': 'gassert_efe4463212c00196c5d5dbc8',
    'claim_text': None,
    'normalized_body': {'predicate': 'oc:hold_command_role',
     'roles': {'commander': [{'entity_id': 'ent:person:journey_cor_a',
        'kind': 'entity'}],
      'organization': [{'entity_id': 'ent:org:journey_cor_org',
        'kind': 'entity'}],
      'title': [{'kind': 'value',
        'value': 'Commander',
        'value_kind': 'string'}]}},
    'predicate': 'oc:hold_command_role',
    'profile': {'profile_id': 'default', 'profile_version': '1.0.0'},
    'promoted_at': '2026-03-18T21:17:01.506674+00:00',
    'promoted_by': 'analyst:graph-promoter',
    'source_candidate_id': 'cand_1418f26bd1a249fc848d2ca9'},
   'entities': [{'created_at': '2026-03-18T21:17:01.506840+00:00',
     'entity_id': 'ent:person:journey_cor_a',
     'entity_type': None,
     'first_candidate_id': 'cand_1418f26bd1a249fc848d2ca9'},
    {'created_at': '2026-03-18T21:17:01.506949+00:00',
     'entity_i

## Artifact Summary

The notebook ends by making the emitted artifacts explicit. This is the key
continuity rule: later phases consume explicit artifacts, not hidden notebook
state.


In [18]:
summary = {
    'journey_id': journey.journey_id,
    'notebook_mode': NOTEBOOK_MODE,
    'artifact_keys': list(artifacts.keys()),
    'current_status': journey.status,
}
pprint(summary)


{'artifact_keys': ['source_artifact_preparation',
                   'candidate_assertion_extraction',
                   'candidate_listing_and_inspection',
                   'candidate_review',
                   'proposal_review',
                   'overlay_application',
                   'overlay_aware_reporting',
                   'domain_pack_generalization',
                   'artifact_lineage_recovery',
                   'epistemic_extension',
                   'product_facing_workflow_integration',
                   'canonical_graph_promotion',
                   'stable_identity_and_external_references',
                   'semantic_canonicalization_repair',
                   'promoted_assertion_epistemics'],
 'current_status': 'proven',
 'journey_id': 'governed_text_to_reviewed_assertions',
 'notebook_mode': 'planning'}


In [19]:
cli_module.TextExtractionService = original_extractor
workspace.cleanup()
